# practice_01 리포트 에이전트 (Colab)

강사 메모 → **1) 주장 뽑기 → 2) 근거 확인 → 3) 강사 질문 → 4) 답변 반영 → 5) 학부모 리포트 → 6) 자가 점검** 을 실제로 돌리는 노트북입니다.

- 설치할 것이 없습니다. 위에서부터 ▶ 버튼을 차례로 누르면 됩니다.
- **1~4번 셀은 mock 모드**라서 외부 API를 부르지 않습니다(비용 0원, 결과가 항상 같음).
- **5번 셀부터 실제 모델**을 부릅니다. OpenRouter 키가 필요하고, 키는 화면에 보이지 않게 입력받으며 어디에도 저장하지 않습니다.
- 모든 메모·답변은 **가짜 데이터**입니다.

## 0. 설정 – 파일 받아오기
PR이 main에 합쳐지기 전에는 `BRANCH = 'toolkit-and-real-agent'` 로 바꿔서 실행하세요.

In [ ]:
BRANCH = 'main'   # 합치기 전이면 'toolkit-and-real-agent'
REPO = 'sejoor12345-commits/hackathon-practice'
import os, urllib.request
FILES = ['practice_01/memos.md', 'practice_01/answers.md', 'practice_01/agent/agent.py',
         'practice_01/eval/memos.md', 'practice_01/eval/answers.md', 'practice_01/eval/gold.json', 'practice_01/eval/grade.py']
for f in FILES:
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{f}'
    os.makedirs(os.path.dirname('/content/' + f), exist_ok=True)
    urllib.request.urlretrieve(url, '/content/' + f)
    print('받음:', f)
import sys
sys.path.insert(0, '/content/practice_01/agent'); sys.path.insert(0, '/content/practice_01/eval')
import importlib, agent, grade
importlib.reload(agent); importlib.reload(grade)
from IPython.display import Markdown, display
def read(p):
    return open('/content/practice_01/' + p, encoding='utf-8').read()

## 1. mock 모드로 원래 메모 10개 돌리기
`load_answers` 는 함수로 넘깁니다. 에이전트는 **질문을 다 만든 뒤에만** 이 함수를 불러 답변을 엽니다.

In [ ]:
r1 = agent.run(read('memos.md'), lambda: read('answers.md'), mode='mock')
display(Markdown(agent.to_markdown(r1)))

## 2. 자가 점검 시연 – 일부러 지어낸 문장 섞기
리포트 3개 중 1개꼴로 "노력 덕분에", "지난 3개월 동안 10점" 같은 문장을 넣습니다. 자가 점검이 몇 개를 빼는지 보세요.

In [ ]:
r2 = agent.run(read('memos.md'), lambda: read('answers.md'), mode='mock', inject_faults=True)
print('넣은 문장:', r2['selfcheck']['injected'], '/ 자가 점검이 뺀 문장:', r2['selfcheck']['injected_caught'])
for rep in r2['reports']:
    for d in rep['dropped']:
        print('-', rep['student'], ':', d['text'], '→', ', '.join(d['problems']))

## 3. 평가 세트(가짜 메모 12개)를 mock 모드로 돌리기

In [ ]:
r3 = agent.run(read('eval/memos.md'), lambda: read('eval/answers.md'), mode='mock')
display(Markdown(agent.to_markdown(r3)))

## 4. 평가 세트 채점
판정·질문·커버리지·지어내기·인용을 셉니다. 이 점수는 **규칙 기반 대역(mock)의 점수**이지 언어 모델 점수가 아닙니다.

In [ ]:
g3 = grade.grade(r3, read('eval/memos.md'), read('eval/answers.md'), __import__('json').loads(read('eval/gold.json')))
display(Markdown(grade.to_markdown(g3, '채점 결과 (mock)')))

## 5. 실제 모델로 돌리기 (OpenRouter)

- 키 만들기: openrouter.ai → 로그인 → Keys → Create Key. **Credit limit(사용 한도)** 을 작게(예: 0이나 1달러) 걸어 두세요.
- 무료 모델(`:free` 또는 `openrouter/free`)은 돈이 들지 않지만 하루 호출 수 제한이 있습니다(OpenRouter 문서 기준, 바뀔 수 있음).
- 평가 세트 12개 1회 실행 ≈ 모델 호출 7회(주장 1 + 판정 1 + 질문 1 + 리포트 3 + 고쳐 쓰기 1).
- 앞 모델이 실패하면(429 등) `MODELS` 의 다음 모델로 넘어갑니다. 무료 모델 이름은 자주 바뀌니 openrouter.ai/models 에서 가격 0인 모델을 찾아 넣으세요.

In [ ]:
import getpass
KEY = getpass.getpass('OpenRouter 키 (화면에 안 보임, 저장 안 함): ')
MODELS = ['openrouter/free']   # 여기에 다른 무료 모델 id 를 쉼표로 더 넣을 수 있음
llm = agent.openrouter_llm(KEY, MODELS)
r5 = agent.run(read('eval/memos.md'), lambda: read('eval/answers.md'), mode='llm', llm=llm,
               on_step=lambda n, i: print('…', n))
r5['llm']['models_used'] = llm.used
print('모델 호출:', r5['llm']['calls'], '/ 실제로 답한 모델:', sorted(set(llm.used)))
print('규칙으로 대신한 곳:', r5['llm']['fallbacks'])
print('모델과 규칙 판정이 다른 메모:', r5['llm']['disagreements'])
display(Markdown(agent.to_markdown(r5)))

## 6. 실제 모델 결과 채점 + 파일로 받기
점수를 `practice_01/eval/RESULTS.md` 의 "실측" 칸에 옮겨 적으세요. 날짜와 모델 이름도 같이 적습니다.

In [ ]:
import json
g5 = grade.grade(r5, read('eval/memos.md'), read('eval/answers.md'), json.loads(read('eval/gold.json')))
display(Markdown(grade.to_markdown(g5, '채점 결과 (실제 모델)')))
with open('/content/llm_eval_run.json', 'w', encoding='utf-8') as f:
    json.dump(r5, f, ensure_ascii=False, indent=2)
try:
    from google.colab import files
    files.download('/content/llm_eval_run.json')
except Exception as e:
    print('Colab 밖이라 내려받기 생략:', e)